# 🔬 GIAI ĐOẠN 1 — CONTROLLED ABLATION STUDY MiT-B0 (M1 → M4) TRÊN COLAB G4 (96 GB BLACKWELL)

> **Mục tiêu Giai đoạn 1 (`One-Factor-At-a-Time`, công bằng tuyệt đối):**
> Chạy độc lập 4 mô hình **`M1, M2, M3, M4`** trên cùng 1 Baseline, cùng `seed = 3407`, cùng ngân sách **`5,000 iters × batch_size 32 = 160,000` mẫu**, và đánh giá **Full-Scan (`stride = 384×384` chuẩn lưới `8×8` DCT)** trên toàn bộ **1,203 ảnh `val.txt`**.
> Sau khi có bảng kết quả thật của `M1–M4` (đặc biệt là `M2` sau khi đã fix lỗi nạp bản đồ **OCR thật**), ta mới quyết định nên giữ/bỏ thành phần nào để kết hợp thành bản tối ưu cuối cùng!
>
> **4 mô hình độc lập:**
> * **`M1` (Baseline 2-Stream):** `RGB + Forensic`, `RandomCrop`, `Memory Bank`, `Head 256ch`.
> * **`M2` (`M1 + Real OCR 3-Stream`):** Cô lập nhánh **OCR thật** (đã fix bug đường dẫn `ocr_spatial` + đồng bộ flip H/V).
> * **`M3` (`M1 + FocusedCrop + Rotate90`):** Cô lập chiến lược lấy mẫu định hướng mẫu dương (`pos_prob=0.65`) & xoay 90° bảo toàn lưới DCT.
> * **`M4` (`M1 + LitePrototypeHead + CE+Dice`):** Cô lập đầu giải mã siêu nhẹ `LitePrototypeHead` (`128ch/64dim`, DW-Sep, giảm **97%** FLOPs Head xuống `5.90 GFLOPs`) + Dual EMA Prototypes + `CE+Dice`.
>
> **Tối ưu tốc độ riêng cho Colab G4 (96 GB VRAM):** Native **BF16 + TF32**, `batch_size=32`, train **song song 2 model/đợt** (đúng 2 đợt gọn cho 4 model), sweep **song song cả 4 model cùng lúc**, lưu trên SSD cục bộ rồi tự động backup sang Google Drive!

### 🟢 BƯỚC 1: KẾT NỐI DRIVE, CÀI MÔI TRƯỜNG & GIẢI NÉN DATASET (~3 Phút)

In [ ]:
import os, sys, glob, torch
from google.colab import drive

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️ GPU: {props.name} | VRAM: {props.total_memory/(1024**3):.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")
    print(f"   CPU cores: {os.cpu_count()} | PyTorch: {torch.__version__}")
else:
    raise RuntimeError("❌ Chưa bật GPU Runtime! Hãy chọn Runtime -> Change runtime type -> G4 GPU.")

drive.mount("/content/drive")

%cd /content
!rm -rf TMD
!git clone https://github.com/gagaduy/TMD.git

sys.path.insert(0, "/content/TMD/ASCFormer")
%env PYTHONPATH=/content/TMD/ASCFormer

!pip install -q yapf==0.40.1 mmcv-lite==2.0.0 mmengine==0.7.0 timm einops prettytable rich onnx onnxruntime

os.makedirs("/content/TMD/ASCFormer/data/ttd", exist_ok=True)
drive_zip = "/content/drive/MyDrive/RealTextMan.zip"
if not os.path.exists(drive_zip):
    matches = glob.glob("/content/drive/MyDrive/**/RealTextMan.zip", recursive=True)
    if matches:
        drive_zip = matches[0]
    else:
        raise FileNotFoundError("❌ Không tìm thấy RealTextMan.zip trên Google Drive!")

print(f"📦 Đang giải nén {drive_zip} sang SSD Colab...")
!unzip -q -o "{drive_zip}" -d /content/TMD/ASCFormer/data/ttd/

ocr_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/ocr_spatial/*.png"))
img_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/JPEGImages/*.jpg"))
print(f"✅ Dataset sẵn sàng: {img_cnt} ảnh JPEGImages, {ocr_cnt} bản đồ ocr_spatial thật!")
assert img_cnt > 0 and ocr_cnt > 0


### 🔥 BƯỚC 2: HUẤN LUYỆN SONG SONG 4 MÔ HÌNH ĐỘC LẬP `M1, M2, M3, M4` TRÊN COLAB G4
> * Chia đẹp thành **2 đợt × 2 model song song** (`Đợt 1: M2 + M1`, `Đợt 2: M3 + M4`) trên SSD cục bộ với **BF16 + TF32** (`5,000 iters × bs=32 = 160,000` mẫu/model).
> * Tự động sao lưu checkpoint & log sang `/content/drive/MyDrive/RTM_Ablation_B0_G4_M1_M4/` ngay khi từng model xong (và tự `--resume` nếu ngắt phiên).
> * **Ước tính tổng thời gian trên G4:** khoảng **~25 – 35 phút** cho cả 4 model!

In [ ]:
import os, sys, glob, time, re, shutil, subprocess, torch

%cd /content/TMD
!git pull origin main
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

# Tải trước Pretrained MiT-B0 để tránh race condition khi chạy song song
_ = torch.hub.load_state_dict_from_url(
    "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth",
    map_location="cpu"
)

DRIVE_ROOT = "/content/drive/MyDrive/RTM_Ablation_B0_G4_M1_M4"
LOCAL_ROOT = "/content/work_dirs/ablation_b0_g4"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
MAX_PARALLEL_TRAIN = 2 if vram_gb >= 60 else 1
print(f"⚡ G4 Scheduler: VRAM={vram_gb:.1f} GB -> MAX_PARALLEL_TRAIN = {MAX_PARALLEL_TRAIN}")

ABLATION_SUITE = [
    ("M2_RealOCR_3Stream",    "configs/ascformer/ablation_b0_h100_m2_ocr.py"),
    ("M1_Baseline_2Stream",   "configs/ascformer/ablation_b0_h100_m1_baseline.py"),
    ("M3_Focused_Rotate90",   "configs/ascformer/ablation_b0_h100_m3_focused.py"),
    ("M4_LiteProto_CEDice",   "configs/ascformer/ablation_b0_h100_m4_liteproto.py"),
]

def backup_to_drive(local_dir, drive_dir):
    os.makedirs(drive_dir, exist_ok=True)
    for ext in ("*.pth", "*.log", "*.json", "*.py"):
        for fp in glob.glob(os.path.join(local_dir, ext)):
            shutil.copy2(fp, os.path.join(drive_dir, os.path.basename(fp)))
    for fp in glob.glob(os.path.join(local_dir, "*", "*.log")):
        shutil.copy2(fp, os.path.join(drive_dir, os.path.basename(fp)))

def parse_progress(log_path):
    if not os.path.exists(log_path):
        return "Đang khởi tạo..."
    try:
        with open(log_path, "r", errors="ignore") as f:
            lines = f.readlines()[-40:]
        for ln in reversed(lines):
            if "Iter(train)" in ln:
                m = re.search(r"Iter\(train\)\s*\[\s*(\d+/\d+)\].*?eta:\s*([0-9:]+).*?loss:\s*([0-9.]+)", ln)
                if m:
                    return f"Iter [{m.group(1)}] | loss={m.group(3)} | ETA={m.group(2)}"
            if "Iter(val)" in ln:
                return "Đang chạy Valid 1,203 ảnh..."
    except Exception:
        pass
    return "Đang nạp dữ liệu / compile..."

queue = []
for name, cfg_path in ABLATION_SUITE:
    d_dir = os.path.join(DRIVE_ROOT, name)
    if os.path.exists(os.path.join(d_dir, "iter_5000.pth")):
        print(f"✅ [{name}] Đã hoàn tất 5,000 iters trên Drive — bỏ qua!")
    else:
        queue.append((name, cfg_path))

running = {}
failed = []
t_global = time.time()

while queue or running:
    while queue and len(running) < MAX_PARALLEL_TRAIN:
        name, cfg_path = queue.pop(0)
        l_dir = os.path.join(LOCAL_ROOT, name)
        d_dir = os.path.join(DRIVE_ROOT, name)
        os.makedirs(l_dir, exist_ok=True)
        os.makedirs(d_dir, exist_ok=True)

        d_ckpts = sorted(glob.glob(os.path.join(d_dir, "iter_*.pth")), key=os.path.getmtime)
        resume_args = []
        if d_ckpts:
            dst = os.path.join(l_dir, os.path.basename(d_ckpts[-1]))
            shutil.copy2(d_ckpts[-1], dst)
            resume_args = ["--resume", dst]

        log_path = os.path.join(l_dir, f"{name}_stdout.log")
        fh = open(log_path, "w")

        runner_code = (
            "import torch, yapf.yapflib.yapf_api as y, mmengine.config.config as m, sys, runpy; "
            "torch.backends.cuda.matmul.allow_tf32 = True; "
            "torch.backends.cudnn.allow_tf32 = True; "
            "torch.backends.cudnn.benchmark = True; "
            "_f = y.FormatCode; "
            "def _sf(*a, **k): k.pop('verify', None); return _f(*a, **k)\n"
            "y.FormatCode = _sf; m.FormatCode = _sf; "
            f"sys.argv = ['tools/train.py', '{cfg_path}', '--work-dir', '{l_dir}'] + {repr(resume_args)}; "
            "runpy.run_path('tools/train.py', run_name='__main__')"
        )
        env = os.environ.copy()
        env["PYTHONPATH"] = "/content/TMD/ASCFormer"
        env["OMP_NUM_THREADS"] = "2"
        proc = subprocess.Popen([sys.executable, "-c", runner_code], stdout=fh, stderr=subprocess.STDOUT, env=env)
        running[name] = (proc, fh, log_path, l_dir, d_dir, time.time())
        print(f"🚀 Khởi chạy [{name}] (PID={proc.pid}) | Resume={resume_args}")

    time.sleep(30)
    elapsed = (time.time() - t_global) / 60.0
    gpu_stat = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"\n⏱️ [{elapsed:.1f}p] G4 GPU (VRAM / Util): {gpu_stat}")

    finished_names = []
    for name, (proc, fh, log_path, l_dir, d_dir, t0) in running.items():
        ret = proc.poll()
        if ret is None:
            print(f"   🔄 {name:<22}: {parse_progress(log_path)}")
        else:
            fh.close()
            dt = (time.time() - t0) / 60.0
            if ret == 0:
                print(f"   🎉 {name:<22}: XONG trong {dt:.1f}p! Đang sao lưu lên Drive...")
                backup_to_drive(l_dir, d_dir)
            else:
                print(f"   ❌ {name:<22}: LỖI (exit={ret})! 20 dòng log cuối:")
                os.system(f"tail -n 20 {log_path}")
                failed.append(name)
            finished_names.append(name)
    for n in finished_names:
        running.pop(n)

assert not failed, f"Có model bị lỗi: {failed}"
print(f"\n🏆 HOÀN TẤT CẢ 4 MÔ HÌNH M1 → M4 TRONG {(time.time()-t_global)/60.0:.1f} PHÚT!")


### 📊 BƯỚC 3: QUÉT ĐA NGƯỠNG FULL-SCAN SONG SONG CẢ 4 MÔ HÌNH (`M1 → M4`, 1,203 ẢNH VAL) & IN BẢNG SO SÁNH
> Chạy song song cả 4 tiến trình sweep cùng lúc trên 96 GB VRAM (~4–6 phút xong cả 4 model!). Sau khi xem bảng kết quả này, bạn gửi lại mình để chốt xem sẽ kết hợp **`M3+M4`** hay **`M2+M3+M4`** cho bước tiếp theo!

In [ ]:
import os, sys, glob, json, time, subprocess
from prettytable import PrettyTable

%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

DRIVE_ROOT = "/content/drive/MyDrive/RTM_Ablation_B0_G4_M1_M4"
LOCAL_ROOT = "/content/work_dirs/ablation_b0_g4"
ABLATION_SUITE = [
    ("M1_Baseline_2Stream",   "configs/ascformer/ablation_b0_h100_m1_baseline.py"),
    ("M2_RealOCR_3Stream",    "configs/ascformer/ablation_b0_h100_m2_ocr.py"),
    ("M3_Focused_Rotate90",   "configs/ascformer/ablation_b0_h100_m3_focused.py"),
    ("M4_LiteProto_CEDice",   "configs/ascformer/ablation_b0_h100_m4_liteproto.py"),
]
THRESHOLDS = ["0.05", "0.10", "0.15", "0.20", "0.25", "0.30", "0.35", "0.40", "0.45", "0.50", "0.55", "0.60"]

# Chạy song song cả 4 tiến trình sweep cùng lúc (~12GB / 96GB VRAM)
MAX_PARALLEL_SWEEP = 4
jobs = []
for name, cfg_path in ABLATION_SUITE:
    d_dir = os.path.join(DRIVE_ROOT, name)
    l_dir = os.path.join(LOCAL_ROOT, name)
    sweep_json = os.path.join(d_dir, f"{name}_sweep.json")
    if os.path.exists(sweep_json):
        print(f"✅ [{name}] Đã có file sweep JSON — bỏ qua!")
        continue
    cands = sorted(glob.glob(f"{l_dir}/best_mIoU_*.pth") + glob.glob(f"{d_dir}/best_mIoU_*.pth"), key=os.path.getmtime)
    if not cands:
        cands = sorted(glob.glob(f"{l_dir}/iter_5000.pth") + glob.glob(f"{d_dir}/iter_5000.pth"), key=os.path.getmtime)
    ckpt = cands[-1]
    jobs.append((name, cfg_path, ckpt, sweep_json))

running = {}
t0 = time.time()
while jobs or running:
    while jobs and len(running) < MAX_PARALLEL_SWEEP:
        name, cfg_path, ckpt, sweep_json = jobs.pop(0)
        cmd = [sys.executable, "tools/test_threshold_sweep.py", cfg_path, ckpt, "--ann-file", "val.txt", "--thresholds", *THRESHOLDS, "--out-json", sweep_json]
        env = os.environ.copy(); env["PYTHONPATH"] = "/content/TMD/ASCFormer"
        log_f = open(f"/tmp/sweep_{name}.log", "w")
        p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
        running[name] = (p, log_f)
        print(f"🔍 Bắt đầu Full-Scan Sweep 1,203 ảnh: [{name}] ({os.path.basename(ckpt)})")
    time.sleep(15)
    done = []
    for name, (p, log_f) in running.items():
        ret = p.poll()
        if ret is not None:
            log_f.close()
            assert ret == 0, f"❌ Sweep lỗi cho {name}! Xem /tmp/sweep_{name}.log"
            print(f"   ✅ Xong Sweep [{name}]!")
            done.append(name)
    for n in done:
        running.pop(n)

print(f"⏱️ Hoàn tất toàn bộ 4 Sweep trong {(time.time()-t0)/60.0:.1f} phút!")

summary = {}
t_std = PrettyTable(["Model", "tau", "Tamper IoU (%)", "mIoU (%)", "Precision (%)", "Recall (%)", "F1 / Dice (%)"])
t_opt = PrettyTable(["Model", "Peak tau*", "Tamper IoU (%)", "mIoU (%)", "Precision (%)", "Recall (%)", "F1 / Dice (%)"])

for name, _ in ABLATION_SUITE:
    sweep_json = os.path.join(DRIVE_ROOT, name, f"{name}_sweep.json")
    res = json.load(open(sweep_json))["results"]
    r50 = [r for r in res if abs(r["threshold"] - 0.50) < 1e-4][0]
    rpk = max(res, key=lambda r: r["tampered_iou"])
    summary[name] = {"tau_050": r50, "peak": rpk}
    t_std.add_row([name, "0.500", f"{r50['tampered_iou']:.2f}", f"{r50['m_iou']:.2f}", f"{r50['tampered_precision']:.2f}", f"{r50['tampered_recall']:.2f}", f"{r50['tampered_dice']:.2f}"])
    t_opt.add_row([name, f"{rpk['threshold']:.3f}", f"{rpk['tampered_iou']:.2f}", f"{rpk['m_iou']:.2f}", f"{rpk['tampered_precision']:.2f}", f"{rpk['tampered_recall']:.2f}", f"{rpk['tampered_dice']:.2f}"])

sum_path = os.path.join(DRIVE_ROOT, "ablation_M1_to_M4_summary.json")
with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n📌 BẢNG ABLATION ĐỘC LẬP M1–M4 TẠI NGƯỠNG CHUẨN (tau = 0.500):")
print(t_std)
print("\n📌 BẢNG ABLATION ĐỘC LẬP M1–M4 TẠI NGƯỠNG TỐI ƯU (Peak tau*):")
print(t_opt)
print("💾 Đã lưu bảng tổng kết vào:", sum_path)
